# Installments Payments Feature Engineering

Here we turn observed payment/schedule records into **one row per applicant in
the previous-application mapping**. We follow the small cells, named Pandas
aggregations, English/Arabic explanations, and saving checks in `01_bureau.ipynb`,
`02_previous_application.ipynb`, and `03_pos_cash.ipynb`.

The decisions come from `../eda/05_installments_payments_eda.ipynb` and are checked
again against the source. Four grains matter:

| Grain | Meaning | المعنى بالعربي |
|---|---|---|
| Raw record | One recorded payment/schedule entry | سجل دفع؛ مش بالضرورة قسط كامل |
| Version-specific installment key | Contract + calendar version + installment number | رقم قسط داخل نسخة محددة من الجدول |
| Previous contract | `SK_ID_PREV` | العقد السابق |
| Applicant | Mapped `SK_ID_CURR` | العميل |

Several entries can contribute to one key. Different versions can describe
revisions of the same obligation, so observed key counts are not independent debts.
No application outcome or `TARGET` is loaded.


## 1. Imports and Paths

We use the same `REPO_ROOT`, `DATA_RAW`, and `DATA_INTERIM` convention as the
Bureau notebook. Searching the current directory and its parents allows execution
from the project root or this notebook directory.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import hashlib
import gc
from collections import Counter
from IPython.display import display, Markdown


In [2]:
# Locate the project without a personal username, drive, or fixed notebook depth.
REPO_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data/raw/home-credit-default-risk").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the project and extract the Kaggle CSVs "
        "into data/raw/home-credit-default-risk/."
    )
DATA_RAW = REPO_ROOT / "data/raw/home-credit-default-risk"
DATA_INTERIM = REPO_ROOT / "data/interim"
DATA_PROCESSED = REPO_ROOT / "data/processed"
INSTALLMENTS_PATH = DATA_RAW / "installments_payments.csv"
PREV_PATH = DATA_RAW / "previous_application.csv"
OUTPUT_PATH = DATA_INTERIM / "installments_features.parquet"
raw_sha256_before = {}
for path in [INSTALLMENTS_PATH, PREV_PATH]:
    with path.open("rb") as raw_file:
        raw_sha256_before[path] = hashlib.file_digest(raw_file, "sha256").hexdigest()
print("Project root:", REPO_ROOT)


Project root: e:\Depi Notes Data Engineering\DEPI project R5\ProJect


## 2. Load the Required Datasets

The EDA reported about 13.6 million rows. Compact IDs and day fields reduce memory;
**amounts remain float64** because rounding money to float32 can distort small
shortfall comparisons. We verify observed day/version values are whole numbers
within float32's exact-integer range.

Only the two mapping IDs are read from `previous_application.csv`. The actual
installment CSV also contains `SK_ID_CURR`; we audit it and then remove it.
It never supplies applicant IDs to the engineered features.


In [3]:
source_columns = pd.read_csv(INSTALLMENTS_PATH, nrows=0).columns.tolist()
KEY = ["SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"]
required_columns = KEY + ["DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"]
assert set(required_columns).issubset(source_columns)
has_source_applicant_id = "SK_ID_CURR" in source_columns
DTYPES = {
    "SK_ID_PREV": "int32", "SK_ID_CURR": "int32", "NUM_INSTALMENT_NUMBER": "int32",
    "NUM_INSTALMENT_VERSION": "float32", "DAYS_INSTALMENT": "float32",
    "DAYS_ENTRY_PAYMENT": "float32", "AMT_INSTALMENT": "float64", "AMT_PAYMENT": "float64"
}
payments = pd.read_csv(
    INSTALLMENTS_PATH,
    usecols=required_columns + (["SK_ID_CURR"] if has_source_applicant_id else []),
    dtype={name: dtype for name, dtype in DTYPES.items() if name in source_columns}
)
previous_mapping = pd.read_csv(
    PREV_PATH, usecols=["SK_ID_PREV", "SK_ID_CURR"],
    dtype={"SK_ID_PREV": "int32", "SK_ID_CURR": "int32"}
)


## 3. Basic Source Inspection

Repeated installment keys are expected because payments can be split. We retain
**every raw row**, including exact duplicates if present. Exact duplicate members
are flagged as ambiguous for cumulative amount comparisons; we do not decide
that they are independent payments or silently delete them.


In [4]:
source_shape = (len(payments), len(source_columns))
n_source_contracts = payments["SK_ID_PREV"].nunique()
print("Source dimensions:", source_shape)
print("Unique previous contracts:", n_source_contracts)
payments.head()


Source dimensions: (13605401, 8)
Unique previous contracts: 997752


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [5]:
payments.info(memory_usage="deep")
display(payments.isna().sum().rename("missing_source_records").to_frame())
display(payments.describe().T)
dictionary = pd.read_csv(DATA_RAW / "HomeCredit_columns_description.csv", encoding="cp1252")
display(dictionary.loc[dictionary["Table"].eq("installments_payments.csv"), ["Row", "Description"]])
del dictionary


<class 'pandas.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int32  
 1   SK_ID_CURR              int32  
 2   NUM_INSTALMENT_VERSION  float32
 3   NUM_INSTALMENT_NUMBER   int32  
 4   DAYS_INSTALMENT         float32
 5   DAYS_ENTRY_PAYMENT      float32
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float32(3), float64(2), int32(3)
memory usage: 519.0 MB


,missing_source_records
SK_ID_PREV,0
SK_ID_CURR,0
NUM_INSTALMENT_VERSION,0
NUM_INSTALMENT_NUMBER,0
DAYS_INSTALMENT,0
DAYS_ENTRY_PAYMENT,2905
AMT_INSTALMENT,0
AMT_PAYMENT,2905


,count,mean,std,min,25%,50%,75%,max
SK_ID_PREV,13605401.0,1.903365e+06,536202.905546,1000001.0,1434191.000,1896520.000,2369094.000,2843499.000
SK_ID_CURR,13605401.0,2.784449e+05,102718.310411,100001.0,189639.000,278685.000,367530.000,456255.000
NUM_INSTALMENT_VERSION,13605401.0,8.566373e-01,1.035216,0.0,0.000,1.000,1.000,178.000
NUM_INSTALMENT_NUMBER,13605401.0,1.887090e+01,26.664067,1.0,4.000,8.000,19.000,277.000
DAYS_INSTALMENT,13605401.0,-1.042270e+03,800.946289,-2922.0,-1654.000,-818.000,-361.000,-1.000
DAYS_ENTRY_PAYMENT,13602496.0,-1.051114e+03,800.585876,-4921.0,-1662.000,-827.000,-370.000,-1.000
AMT_INSTALMENT,13605401.0,1.705091e+04,50570.254429,0.0,4226.085,8884.080,16710.210,3771487.845
AMT_PAYMENT,13602496.0,1.723822e+04,54735.783981,0.0,3398.265,8125.515,16108.425,3771487.845


,Row,Description
211,SK_ID_PREV,ID of previous credit in Home credit related t...
212,SK_ID_CURR,ID of loan in our sample
213,NUM_INSTALMENT_VERSION,Version of installment calendar (0 is for cred...
214,NUM_INSTALMENT_NUMBER,On which installment we observe payment
215,DAYS_INSTALMENT,When the installment of previous credit was su...
216,DAYS_ENTRY_PAYMENT,When was the installments of previous credit p...
217,AMT_INSTALMENT,What was the prescribed installment amount of ...
218,AMT_PAYMENT,What the client actually paid on previous cred...


In [6]:
assert payments["SK_ID_PREV"].notna().all()
complete_key = payments[KEY].notna().all(axis=1)
missing_key_records = int((~complete_key).sum())
exact_duplicate_extras = int(payments.duplicated().sum())
exact_duplicate_members = payments.duplicated(keep=False)
key_duplicate_extras = int(payments.loc[complete_key, KEY].duplicated().sum())
n_source_keys = int(complete_key.sum()) - key_duplicate_extras
display(pd.Series({
    "Complete version-specific installment keys": n_source_keys,
    "Extra records beyond the first at complete keys": key_duplicate_extras,
    "Missing-key records retained in record features": missing_key_records,
    "Exact duplicate extra records retained": exact_duplicate_extras
}, name="Count").to_frame())


,Count
Complete version-specific installment keys,12951918
Extra records beyond the first at complete keys,653483
Missing-key records retained in record features,0
Exact duplicate extra records retained,0


Both day columns are offsets relative to the current application. Positive days
or the `365243` sentinel would require an availability decision before creating
repayment features. We stop for review if either appears; observed nonpositive
days are usable, and missing days remain unknown.

Version 0 is allowed; negative versions and nonpositive installment numbers
require review. Negative amounts, if encountered, remain in raw inspections and
are excluded from eligible cumulative comparisons later.


In [7]:
for column in ["DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "NUM_INSTALMENT_VERSION"]:
    values = payments[column].dropna()
    assert np.isfinite(values).all(), column
    assert values.mod(1).eq(0).all(), column
    assert values.abs().lt(2**24).all(), "Compact values must remain exact integers."
for column in ["DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]:
    assert payments[column].dropna().le(0).all(), "Post-application days require leakage review."
    assert not payments[column].eq(365243).any()
assert payments["NUM_INSTALMENT_VERSION"].dropna().ge(0).all()
assert payments["NUM_INSTALMENT_NUMBER"].gt(0).all()
for column in ["AMT_INSTALMENT", "AMT_PAYMENT"]:
    assert np.isfinite(payments[column].dropna()).all()
    print(column, "negative:", int(payments[column].lt(0).sum()), "zero:", int(payments[column].eq(0).sum()))
del values
validation_results = {"Source schema, IDs, numeric domains, and application-time offsets": "PASS"}


AMT_INSTALMENT negative: 0 zero: 290
AMT_PAYMENT negative: 0 zero: 1440


## 4. Understand the Contract-to-Applicant Mapping

We validate one applicant per previous ID and compare the optional source
applicant ID at **contract grain**. The final feature population includes all
applicants in the previous mapping, consistent with the previous two notebooks.
Unmatched histories will remain in a separate contract summary with missing mapped IDs.


In [8]:
assert previous_mapping.notna().all().all()
mapping_conflicts = int(previous_mapping.groupby("SK_ID_PREV")["SK_ID_CURR"].nunique().gt(1).sum())
assert mapping_conflicts == 0
assert previous_mapping["SK_ID_PREV"].is_unique, "Review repeated previous mapping IDs."
previous_index = previous_mapping.set_index("SK_ID_PREV")
source_records_per_contract = payments.groupby("SK_ID_PREV").size()
source_ids = source_records_per_contract.index
matched_ids = source_ids.intersection(previous_index.index)
unmatched_ids = source_ids.difference(previous_index.index)
previous_without_history = previous_index.index.difference(source_ids)
n_matched_records = int(source_records_per_contract.loc[matched_ids].sum())
n_unmatched_records = int(source_records_per_contract.loc[unmatched_ids].sum())
n_applicants = previous_mapping["SK_ID_CURR"].nunique()
n_mapped_history_applicants = previous_index.loc[matched_ids, "SK_ID_CURR"].nunique()


In [9]:
n_source_applicants = None
if has_source_applicant_id:
    assert payments["SK_ID_CURR"].notna().all()
    source_id_groups = payments.groupby("SK_ID_PREV")["SK_ID_CURR"]
    assert source_id_groups.nunique().le(1).all()
    source_applicant_map = source_id_groups.first()
    assert source_applicant_map.loc[matched_ids].eq(previous_index.loc[matched_ids, "SK_ID_CURR"]).all()
    n_source_applicants = payments["SK_ID_CURR"].nunique()
    del source_id_groups, source_applicant_map
    payments = payments.drop(columns="SK_ID_CURR")
linkage_summary = pd.Series({
    "Matched installment contracts": len(matched_ids), "Unmatched installment contracts": len(unmatched_ids),
    "Matched raw records": n_matched_records, "Unmatched raw records": n_unmatched_records,
    "Applicants with mapped installment history": n_mapped_history_applicants,
    "Final previous-mapping applicant population": n_applicants
}, name="Count")
display(linkage_summary.to_frame())
assert len(matched_ids) + len(unmatched_ids) == n_source_contracts
assert n_matched_records + n_unmatched_records == source_shape[0]
assert len(matched_ids) + len(previous_without_history) == len(previous_mapping)
validation_results["Mapping conflicts, source-ID agreement, and matched/unmatched conservation"] = "PASS"


,Count
Matched installment contracts,958905
Unmatched installment contracts,38847
Matched raw records,12354575
Unmatched raw records,1250826
Applicants with mapped installment history,336935
Final previous-mapping applicant population,338857


**Coverage interpretation:** Contract ID match coverage is a portfolio audit,
not a constant applicant feature. Applicant history coverage later means mapped
installment contracts / all previous IDs. An application without payment records
may not have created a repayment obligation; it is not proof of nonpayment.

عدم وجود سجل أقساط مش معناه إن العميل ما دفعش؛ ممكن الطلب ما اتحولش لقرض.


## 5. Calculate and Inspect Payment Timing

`days_late = DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT` describes an **entry**:
negative is early, zero is on time, positive is late. Missing dates cannot be
classified as on time. Several late entries can belong to the same installment key.

Signed average delay includes early entries. We separately retain positive-only
delay totals/maxima so early payments cannot cancel late-payment severity.
Thresholds ≥30, ≥60, and ≥90 days are analytical choices, not official default labels.


In [10]:
payments["days_late"] = payments["DAYS_ENTRY_PAYMENT"] - payments["DAYS_INSTALMENT"]
source_timing_audit = {
    "timed_records": int(payments["days_late"].count()),
    "early_records": int(payments["days_late"].lt(0).sum()),
    "ontime_records": int(payments["days_late"].eq(0).sum()),
    "late_records": int(payments["days_late"].gt(0).sum()),
    "delay_sum": payments["days_late"].astype("float64").sum(min_count=1),
    "severe30_records": int(payments["days_late"].ge(30).sum()),
    "severe60_records": int(payments["days_late"].ge(60).sum()),
    "severe90_records": int(payments["days_late"].ge(90).sum())
}
display(pd.Series(source_timing_audit).to_frame("Source record timing"))
assert sum(source_timing_audit[name] for name in ["early_records", "ontime_records", "late_records"]) == source_timing_audit["timed_records"]
timing_sample = payments.sample(30, random_state=42)
np.testing.assert_allclose(timing_sample["days_late"], timing_sample["DAYS_ENTRY_PAYMENT"] - timing_sample["DAYS_INSTALMENT"], equal_nan=True)
validation_results["Source entry delay formula and eligible timing classification"] = "PASS"


,Source record timing
timed_records,13602496.0
early_records,9309477.0
ontime_records,3146350.0
late_records,1146669.0
delay_sum,-119530648.0
severe30_records,43602.0
severe60_records,16352.0
severe90_records,12820.0


## 6. Identify and Document Installment-Level Keys

We use (`SK_ID_PREV`, `NUM_INSTALMENT_VERSION`, `NUM_INSTALMENT_NUMBER`). Missing-key
records stay in record-level counts/timing/amount summaries but cannot enter a
complete scheduled key. The current EDA found no missing keys; we confirm the count.

Version 0 denotes **credit-card calendars**. Its installment numbers are often
not chronological. We use dates directly, retain version 0, and report its timing
separately. We do not choose a “latest version” from the largest version number.
Versions are identifiers, not risk grades or guaranteed chronological revisions.


In [11]:
display(payments["NUM_INSTALMENT_VERSION"].value_counts(dropna=False).sort_index().rename("raw_records").to_frame())
display(payments.loc[payments.duplicated(KEY, keep=False), required_columns + ["days_late"]].head(8))


,raw_records
NUM_INSTALMENT_VERSION,
0.0,4082498
1.0,8485004
2.0,620283
3.0,237063
4.0,55274
...,...
61.0,8
68.0,1
72.0,7


,SK_ID_PREV,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,days_late
4,2714724,1.0,2,-1383.0,-1366.0,2165.040,2160.585,17.0
17,1410565,1.0,1,-1037.0,-1048.0,12514.050,12510.450,-11.0
30,1840786,1.0,11,-476.0,-474.0,21179.700,21161.700,2.0
40,2270983,0.0,30,-2022.0,-2046.0,9000.000,1800.000,-24.0
47,1895324,1.0,6,-1386.0,-1379.0,3229.065,3212.865,7.0
48,2411245,1.0,1,-2588.0,-2555.0,3294.900,650.520,33.0
50,1220449,1.0,16,-61.0,-89.0,34717.500,2675.745,-28.0
52,2726751,1.0,20,-1066.0,-1091.0,10770.750,499.500,-25.0


## 7. Aggregate Records to Installment-Key Level

We process contract partitions to avoid holding a second thirteen-million-row
table. `SK_ID_PREV % 24` keeps **all entries and versions of each contract together**.
Partition number has no feature meaning. Small helpers below each implement one
explained operation; the overall workflow remains visible across separate cells.

`groupby(...).agg()` creates one row per version-specific key. `size` counts all
entries, `count` excludes missing values, and `nunique` checks schedule consistency.
We require every scheduled date/amount to be present and consistent before using
one schedule value. Conflicting keys stay counted and are excluded from comparisons.


In [12]:
N_PARTITIONS = 24
AMOUNT_TOLERANCE = 1e-6  # Dataset units; floating-point summation tolerance only.

def aggregate_installment_keys(part):
    grouped = part.groupby(KEY, sort=False, observed=True)
    keys = grouped.agg(
        records=("SK_ID_PREV", "size"), due_count=("DAYS_INSTALMENT", "count"),
        due_distinct=("DAYS_INSTALMENT", "nunique"), due_day=("DAYS_INSTALMENT", "min"),
        scheduled_count=("AMT_INSTALMENT", "count"), scheduled_distinct=("AMT_INSTALMENT", "nunique"),
        scheduled_amount=("AMT_INSTALMENT", "min"), paid_count=("AMT_PAYMENT", "count"),
        paid_min=("AMT_PAYMENT", "min"), timed_records=("days_late", "count"),
        late_records=("is_late", "sum"), duplicate_members=("exact_duplicate", "sum"),
        positive_payment_missing_date=("positive_payment_missing_date", "sum")
    )
    keys["recorded_payment_sum"] = grouped["AMT_PAYMENT"].sum(min_count=1)
    keys["coherent_schedule"] = (
        keys["due_count"].eq(keys["records"]) & keys["due_distinct"].eq(1)
        & keys["scheduled_count"].eq(keys["records"]) & keys["scheduled_distinct"].eq(1)
    )
    keys["has_multiple_entries"] = keys["records"].gt(1)
    keys["additional_entries"] = keys["records"] - 1
    assert keys.index.is_unique
    assert keys["records"].sum() == len(part)
    return keys


### A. Cumulative Recorded Amounts Versus One Schedule

Eligibility requires a coherent date/amount, **positive** scheduled amount,
complete nonnegative payments, and no exact duplicate members. An all-missing
payment sum stays missing because `sum(min_count=1)` requires an observed value.

We classify below/equal/above with a `1e-6` dataset-unit tolerance. Coverage means
equal or above. This is **observed cumulative coverage**, not final settlement.
Incomplete history, allocation rules, and revisions remain unresolved.

لو القسط 100 واتدفع 40 ثم 60، المجموع يغطي القسط؛ أول دفعة لوحدها مش عجز نهائي.


In [13]:
def add_key_amount_comparisons(keys):
    eligible = (
        keys["coherent_schedule"] & keys["scheduled_amount"].gt(0)
        & keys["paid_count"].eq(keys["records"]) & keys["paid_min"].ge(0)
        & keys["duplicate_members"].eq(0)
    )
    gap = keys["recorded_payment_sum"] - keys["scheduled_amount"]
    keys["amount_eligible"] = eligible
    keys["amount_below"] = eligible & gap.lt(-AMOUNT_TOLERANCE)
    keys["amount_equal"] = eligible & gap.abs().le(AMOUNT_TOLERANCE)
    keys["amount_above"] = eligible & gap.gt(AMOUNT_TOLERANCE)
    keys["amount_covered"] = keys["amount_equal"] | keys["amount_above"]
    keys["comparison_scheduled_amount"] = keys["scheduled_amount"].where(eligible)
    keys["comparison_payment_amount"] = keys["recorded_payment_sum"].where(eligible)
    keys["coherent_scheduled_amount"] = keys["scheduled_amount"].where(
        keys["coherent_schedule"] & keys["scheduled_amount"].ge(0)
    )
    keys["payment_to_schedule_ratio"] = keys["recorded_payment_sum"].where(eligible) / keys["scheduled_amount"].where(eligible)
    assert (keys["amount_below"].astype("int8") + keys["amount_equal"].astype("int8") + keys["amount_above"].astype("int8")).eq(eligible.astype("int8")).all()
    return keys


### B. First Observed Cumulative Coverage Date

For eligible keys, every **positive** payment must have a known date. Zero payments
cannot advance cumulative coverage. We sort positive entries by payment date
within the key, use `cumsum()`, and select the first date reaching the schedule.
Missing or insufficient coverage has no known coverage date. A later extra entry
does not necessarily make an installment's observed coverage late.

`MultiIndex.reindex()` looks up each row's key eligibility. This is a lookup of
key summaries, not a merge of raw records into applicant data.


In [14]:
def add_observed_coverage_timing(part, keys):
    eligible = keys["amount_eligible"] & keys["positive_payment_missing_date"].eq(0)
    row_keys = pd.MultiIndex.from_frame(part[KEY])
    row_eligible = eligible.reindex(row_keys).to_numpy()
    dated = part.loc[row_eligible & part["AMT_PAYMENT"].gt(0),
                     KEY + ["DAYS_ENTRY_PAYMENT", "AMT_PAYMENT", "AMT_INSTALMENT"]].copy()
    dated = dated.sort_values(KEY + ["DAYS_ENTRY_PAYMENT"])
    cumulative = dated.groupby(KEY, sort=False, observed=True)["AMT_PAYMENT"].cumsum()
    reaches_schedule = cumulative.ge(dated["AMT_INSTALMENT"] - AMOUNT_TOLERANCE)
    first_coverage = dated.loc[reaches_schedule].groupby(KEY, sort=False)["DAYS_ENTRY_PAYMENT"].min()
    keys["coverage_day"] = first_coverage.reindex(keys.index)
    keys["coverage_delay"] = keys["coverage_day"] - keys["due_day"]
    keys["coverage_timed"] = keys["coverage_delay"].notna()
    keys["coverage_early"] = keys["coverage_delay"].lt(0)
    keys["coverage_ontime"] = keys["coverage_delay"].eq(0)
    keys["coverage_late"] = keys["coverage_delay"].gt(0)
    keys["coverage_late_delay"] = keys["coverage_delay"].where(keys["coverage_late"])
    assert keys["coverage_timed"].le(eligible & keys["amount_covered"]).all()
    assert (keys["coverage_early"].astype("int8") + keys["coverage_ontime"].astype("int8") + keys["coverage_late"].astype("int8")).eq(keys["coverage_timed"].astype("int8")).all()
    return keys


### Walk Through a Small Illustrative Example

These invented rows teach the calculation and **never enter the feature table**.
Key 1 has a 40 + 60 split payment. Key 2 covers its schedule early, then records
an extra late payment. Key 3 has incomplete payment information. Version 2 of
installment 1 remains a separate observed schedule key.


In [15]:
example_records = pd.DataFrame([
    [1, 1, 1, -10, -12, 100.0, 40.0], [1, 1, 1, -10, -9, 100.0, 60.0],
    [1, 1, 2, -10, -12, 100.0, 100.0], [1, 1, 2, -10, -9, 100.0, 5.0],
    [1, 1, 3, -10, -12, 100.0, 40.0], [1, 1, 3, -10, np.nan, 100.0, np.nan],
    [1, 2, 1, -8, -8, 80.0, 80.0]
], columns=required_columns)
example_records["days_late"] = example_records["DAYS_ENTRY_PAYMENT"] - example_records["DAYS_INSTALMENT"]
example_records["is_late"] = example_records["days_late"].gt(0)
example_records["exact_duplicate"] = False
example_records["positive_payment_missing_date"] = example_records["AMT_PAYMENT"].gt(0) & example_records["DAYS_ENTRY_PAYMENT"].isna()
example_keys = aggregate_installment_keys(example_records)
example_keys = add_key_amount_comparisons(example_keys)
example_keys = add_observed_coverage_timing(example_records, example_keys)
display(example_records[required_columns + ["days_late"]])
display(example_keys[["records", "scheduled_amount", "recorded_payment_sum", "amount_eligible", "coverage_day", "coverage_delay", "late_records"]])


,SK_ID_PREV,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,days_late
0,1,1,1,-10,-12.0,100.0,40.0,-2.0
1,1,1,1,-10,-9.0,100.0,60.0,1.0
2,1,1,2,-10,-12.0,100.0,100.0,-2.0
3,1,1,2,-10,-9.0,100.0,5.0,1.0
4,1,1,3,-10,-12.0,100.0,40.0,-2.0
5,1,1,3,-10,NaN,100.0,NaN,NaN
6,1,2,1,-8,-8.0,80.0,80.0,0.0


records  \
SK_ID_PREV NUM_INSTALMENT_VERSION NUM_INSTALMENT_NUMBER            
1          1                      1                            2   
                                  2                            2   
                                  3                            2   
           2                      1                            1   

                                                         scheduled_amount  \
SK_ID_PREV NUM_INSTALMENT_VERSION NUM_INSTALMENT_NUMBER                     
1          1                      1                                 100.0   
                                  2                                 100.0   
                                  3                                 100.0   
           2                      1                                  80.0   

                                                         recorded_payment_sum  \
SK_ID_PREV NUM_INSTALMENT_VERSION NUM_INSTALMENT_NUMBER                         
1          1                      1                                     100.0   
                                  2                                     105.0   
                                  3                                      40.0   
           2                      1                                      80.0   

                                                         amount_eligible  \
SK_ID_PREV NUM_INSTALMENT_VERSION NUM_INSTALMENT_NUMBER                    
1          1                      1                                 True   
                                  2                                 True   
                                  3                                False   
           2                      1                                 True   

                                                         coverage_day  \
SK_ID_PREV NUM_INSTALMENT_VERSION NUM_INSTALMENT_NUMBER                 
1          1                      1                              -9.0   
                                  2                             -12.0   
                                  3                               NaN   
           2                      1                              -8.0   

                                                         coverage_delay  \
SK_ID_PREV NUM_INSTALMENT_VERSION NUM_INSTALMENT_NUMBER                   
1          1                      1                                 1.0   
                                  2                                -2.0   
                                  3                                 NaN   
           2                      1                                 0.0   

                                                         late_records  
SK_ID_PREV NUM_INSTALMENT_VERSION NUM_INSTALMENT_NUMBER                
1          1                      1                                 1  
                                  2                                 1  
                                  3                                 0  
           2                      1                                 0

In [16]:
assert example_keys.loc[(1, 1, 1), "recorded_payment_sum"] == 100
assert example_keys.loc[(1, 1, 1), "amount_equal"]
assert example_keys.loc[(1, 1, 1), "coverage_day"] == -9
assert example_keys.loc[(1, 1, 2), "amount_above"]
assert example_keys.loc[(1, 1, 2), "coverage_day"] == -12
assert example_keys.loc[(1, 1, 2), "late_records"] == 1
assert not example_keys.loc[(1, 1, 3), "amount_eligible"]
assert pd.isna(example_keys.loc[(1, 1, 3), "coverage_day"])
assert len(example_keys) == 4
validation_results["Illustrative split-payment, late-extra-entry, missingness, and version boundaries"] = "PASS"
del example_records, example_keys


## 8. Engineer Contract-Level Features

### A. Payment-Record Counts, Timing, Severity, and History

Raw record features are valid at record grain: observed entry counts, signed and
positive-only delays, missing fields, and payment-date endpoints. They are first
aggregated to `SK_ID_PREV` before any applicant features are created.

We keep day totals in float64 for accurate weighted means. Actual zero payment
amounts stay zero; `sum(min_count=1)` keeps an all-missing payment total missing.


In [17]:
record_grouped = payments.groupby("SK_ID_PREV", sort=True)
payments["delay_for_sum"] = payments["days_late"].astype("float64")
payments["entry_day_for_sum"] = payments["DAYS_ENTRY_PAYMENT"].astype("float64")
record_features = record_grouped.agg(
    records=("SK_ID_PREV", "size"), timed_records=("days_late", "count"),
    delay_max=("days_late", "max"), payment_date_records=("DAYS_ENTRY_PAYMENT", "count"),
    due_date_records=("DAYS_INSTALMENT", "count"), payment_amount_records=("AMT_PAYMENT", "count"),
    scheduled_amount_records=("AMT_INSTALMENT", "count"),
    oldest_payment_day=("DAYS_ENTRY_PAYMENT", "min"), latest_payment_day=("DAYS_ENTRY_PAYMENT", "max"),
    oldest_due_day=("DAYS_INSTALMENT", "min"), latest_due_day=("DAYS_INSTALMENT", "max"),
    observed_versions=("NUM_INSTALMENT_VERSION", "nunique")
)
record_features["delay_sum"] = record_grouped["delay_for_sum"].sum(min_count=1)
record_features["payment_day_sum"] = record_grouped["entry_day_for_sum"].sum(min_count=1)
record_features["recorded_payment_sum"] = record_grouped["AMT_PAYMENT"].sum(min_count=1)
source_payment_sum = payments["AMT_PAYMENT"].sum(min_count=1)
del payments["delay_for_sum"], payments["entry_day_for_sum"]


In [18]:
for name, event in {
    "early_records": payments["days_late"].lt(0), "ontime_records": payments["days_late"].eq(0),
    "late_records": payments["days_late"].gt(0), "severe30_records": payments["days_late"].ge(30),
    "severe60_records": payments["days_late"].ge(60), "severe90_records": payments["days_late"].ge(90),
    "missing_key_records": ~complete_key
}.items():
    record_features[name] = event.groupby(payments["SK_ID_PREV"]).sum().astype("int32")
payments["late_delay"] = payments["days_late"].where(payments["days_late"].gt(0)).astype("float64")
payments["late_payment_day"] = payments["DAYS_ENTRY_PAYMENT"].where(payments["days_late"].gt(0))
record_features["late_delay_sum"] = record_grouped["late_delay"].sum(min_count=1)
record_features["late_delay_max"] = record_grouped["late_delay"].max()
record_features["latest_late_payment_day"] = record_grouped["late_payment_day"].max()
for name, observed_count in {
    "missing_timing_records": "timed_records", "missing_payment_dates": "payment_date_records",
    "missing_due_dates": "due_date_records", "missing_payment_amounts": "payment_amount_records",
    "missing_scheduled_amounts": "scheduled_amount_records"
}.items():
    record_features[name] = record_features["records"] - record_features[observed_count]
del payments["late_delay"], payments["late_payment_day"]
del event


### B. Separate Version-0 and Positive-Version Timing

These counts describe product/calendar mix. Timing is calculated from dates in
both groups; no installment-number ordering is assumed. Version-0 and positive-
version rates later use their own known-timing denominators.


In [19]:
for label, version_mask in {
    "version0": payments["NUM_INSTALMENT_VERSION"].eq(0),
    "positive_version": payments["NUM_INSTALMENT_VERSION"].gt(0)
}.items():
    record_features[label + "_records"] = version_mask.groupby(payments["SK_ID_PREV"]).sum()
    record_features[label + "_timed_records"] = (version_mask & payments["days_late"].notna()).groupby(payments["SK_ID_PREV"]).sum()
    record_features[label + "_late_records"] = (version_mask & payments["days_late"].gt(0)).groupby(payments["SK_ID_PREV"]).sum()
for column in record_features.select_dtypes(include="integer").columns:
    record_features[column] = record_features[column].astype("int32")
assert record_features.index.is_unique
assert len(record_features) == n_source_contracts
assert record_features["records"].sum() == source_shape[0]
for name, expected in source_timing_audit.items():
    np.testing.assert_allclose(record_features[name].sum(min_count=1), expected, rtol=1e-12)
assert record_features[["early_records", "ontime_records", "late_records"]].sum(axis=1).eq(record_features["timed_records"]).all()
assert record_features["late_records"].le(record_features["timed_records"]).all()
np.testing.assert_allclose(record_features["recorded_payment_sum"].sum(min_count=1), source_payment_sum, rtol=1e-12)
validation_results["Unique record-summary contract grain and source timing/amount conservation"] = "PASS"
del record_grouped, version_mask


### C. Contract Summaries of Installment Keys

We count keys, multiple entries, eligible comparisons, and observed coverage timing
**inside each contract**. Scheduled amount is counted once per coherent key, not
once per payment row. Its total remains an **observed version-key schedule total**;
revisions can repeat an obligation, so it is not contractual debt or principal.

Comparison payment and scheduled totals use exactly the same eligible keys.
Raw recorded-payment totals remain a separate feature and can include incomplete keys.


In [20]:
def summarize_keys_by_contract(keys):
    keys["has_timing"] = keys["timed_records"].gt(0)
    keys["has_late_record"] = keys["late_records"].gt(0)
    keys["version0_key"] = keys.index.get_level_values("NUM_INSTALMENT_VERSION") == 0
    keys["inconsistent_schedule"] = keys["due_distinct"].gt(1) | keys["scheduled_distinct"].gt(1)
    keys["incomplete_schedule"] = keys["due_count"].ne(keys["records"]) | keys["scheduled_count"].ne(keys["records"])
    keys["incomplete_payments"] = keys["paid_count"].ne(keys["records"])
    keys["zero_schedule"] = keys["coherent_schedule"] & keys["scheduled_amount"].eq(0)
    grouped = keys.groupby(level="SK_ID_PREV", sort=True)
    result = grouped.agg(
        schedule_keys=("records", "size"), key_records=("records", "sum"),
        multiple_entry_keys=("has_multiple_entries", "sum"), additional_entries=("additional_entries", "sum"),
        timed_keys=("has_timing", "sum"), keys_with_late_records=("has_late_record", "sum"),
        amount_eligible_keys=("amount_eligible", "sum"), amount_below_keys=("amount_below", "sum"),
        amount_equal_keys=("amount_equal", "sum"), amount_above_keys=("amount_above", "sum"),
        amount_covered_keys=("amount_covered", "sum"), coherent_amount_keys=("coherent_scheduled_amount", "count"),
        scheduled_key_sum=("coherent_scheduled_amount", "sum"),
        comparison_scheduled_sum=("comparison_scheduled_amount", "sum"), comparison_payment_sum=("comparison_payment_amount", "sum"),
        coverage_timed_keys=("coverage_timed", "sum"), coverage_early_keys=("coverage_early", "sum"),
        coverage_ontime_keys=("coverage_ontime", "sum"), coverage_late_keys=("coverage_late", "sum"),
        late_coverage_delay_sum=("coverage_late_delay", "sum"), late_coverage_delay_max=("coverage_late_delay", "max"),
        version0_keys=("version0_key", "sum"), inconsistent_schedule_keys=("inconsistent_schedule", "sum"),
        incomplete_schedule_keys=("incomplete_schedule", "sum"), incomplete_payment_keys=("incomplete_payments", "sum"),
        zero_schedule_keys=("zero_schedule", "sum")
    )
    for column, count in {"scheduled_key_sum": "coherent_amount_keys", "comparison_scheduled_sum": "amount_eligible_keys", "comparison_payment_sum": "amount_eligible_keys", "late_coverage_delay_sum": "coverage_late_keys"}.items():
        result[column] = result[column].mask(result[count].eq(0))
    for column in result.select_dtypes(include="integer").columns:
        result[column] = result[column].astype("int32")
    return result


### D. Repeated Installment Numbers Across Versions

A second grouping on `(SK_ID_PREV, NUM_INSTALMENT_NUMBER)` counts observed
versions for the same number. We report cross-version pairs and changed scheduled
amount/date counts only when every version has a coherent schedule. These are
revision-context indicators, not confirmation of rescheduling or extra debts.


In [21]:
def summarize_revision_pairs(keys):
    numbered = keys.reset_index().groupby(["SK_ID_PREV", "NUM_INSTALMENT_NUMBER"]).agg(
        versions=("NUM_INSTALMENT_VERSION", "nunique"), coherent=("coherent_schedule", "all"),
        amount_min=("scheduled_amount", "min"), amount_max=("scheduled_amount", "max"),
        due_min=("due_day", "min"), due_max=("due_day", "max")
    )
    numbered["cross_version"] = numbered["versions"].gt(1)
    numbered["comparable_revision"] = numbered["cross_version"] & numbered["coherent"]
    numbered["changed_amount"] = numbered["comparable_revision"] & numbered["amount_max"].gt(numbered["amount_min"])
    numbered["changed_due"] = numbered["comparable_revision"] & numbered["due_max"].gt(numbered["due_min"])
    return numbered.groupby(level="SK_ID_PREV").agg(
        cross_version_number_pairs=("cross_version", "sum"),
        comparable_revision_pairs=("comparable_revision", "sum"),
        changed_amount_revision_pairs=("changed_amount", "sum"),
        changed_due_revision_pairs=("changed_due", "sum")
    ).astype("int32")


### Run the Key Calculations in Contract Partitions

The loop follows the helpers in order: aggregate entries, compare cumulative
amounts, find first coverage dates, summarize contracts, and inspect revisions.
It retains only compact contract summaries and a few real source examples.
Assertions run in every partition; no key or contract crosses partitions.


In [22]:
key_contract_parts = []
key_audit = Counter()
checked_key_parts = []
check_contract_ids = record_features.sample(12, random_state=42).index.tolist()
for column in ["severe90_records", "missing_timing_records", "missing_payment_amounts", "missing_scheduled_amounts", "version0_records"]:
    check_contract_ids += record_features.loc[record_features[column].gt(0)].head(2).index.tolist()
partition_codes = payments["SK_ID_PREV"].mod(N_PARTITIONS).astype("int8")


In [23]:
for partition in range(N_PARTITIONS):
    selected = partition_codes.eq(partition) & complete_key
    part = payments.loc[selected, required_columns + ["days_late"]].copy()
    part["exact_duplicate"] = exact_duplicate_members.loc[selected].to_numpy()
    part["is_late"] = part["days_late"].gt(0)
    part["positive_payment_missing_date"] = part["AMT_PAYMENT"].gt(0) & part["DAYS_ENTRY_PAYMENT"].isna()
    keys = aggregate_installment_keys(part)
    keys = add_key_amount_comparisons(keys)
    keys = add_observed_coverage_timing(part, keys)
    result = summarize_keys_by_contract(keys).join(summarize_revision_pairs(keys), validate="one_to_one")
    key_contract_parts.append(result)
    key_audit["keys"] += len(keys)
    key_audit["represented_records"] += int(keys["records"].sum())
    for column in ["multiple_entry_keys", "additional_entries", "amount_eligible_keys", "amount_below_keys", "amount_equal_keys", "amount_above_keys", "amount_covered_keys", "coverage_timed_keys", "coverage_early_keys", "coverage_ontime_keys", "coverage_late_keys", "cross_version_number_pairs", "inconsistent_schedule_keys", "incomplete_schedule_keys", "incomplete_payment_keys", "zero_schedule_keys"]:
        key_audit[column] += int(result[column].sum())
    key_audit["late_entries_despite_nonlate_coverage"] += int((keys["coverage_timed"] & keys["coverage_delay"].le(0) & keys["late_records"].gt(0)).sum())
    if partition == 0:
        for mask in [keys["has_multiple_entries"], keys["amount_below"], ~keys["amount_eligible"], keys["coverage_timed"] & keys["coverage_delay"].le(0) & keys["late_records"].gt(0)]:
            check_contract_ids += keys.index.get_level_values("SK_ID_PREV")[mask][:2].tolist()
    checked_key_parts.append(keys.loc[keys.index.get_level_values("SK_ID_PREV").isin(check_contract_ids)].copy())
    print(f"Processed {partition + 1}/{N_PARTITIONS} partitions; {key_audit['keys']:,} keys", flush=True)
    del part, keys, result
    gc.collect()


Processed 1/24 partitions; 540,387 keys
Processed 2/24 partitions; 1,074,938 keys
Processed 3/24 partitions; 1,612,138 keys
Processed 4/24 partitions; 2,152,454 keys
Processed 5/24 partitions; 2,691,241 keys
Processed 6/24 partitions; 3,222,275 keys
Processed 7/24 partitions; 3,764,066 keys
Processed 8/24 partitions; 4,310,113 keys
Processed 9/24 partitions; 4,853,408 keys
Processed 10/24 partitions; 5,391,750 keys
Processed 11/24 partitions; 5,937,751 keys
Processed 12/24 partitions; 6,478,173 keys
Processed 13/24 partitions; 7,018,417 keys
Processed 14/24 partitions; 7,555,362 keys
Processed 15/24 partitions; 8,093,117 keys
Processed 16/24 partitions; 8,631,504 keys
Processed 17/24 partitions; 9,171,122 keys
Processed 18/24 partitions; 9,709,017 keys
Processed 19/24 partitions; 10,244,421 keys
Processed 20/24 partitions; 10,788,135 keys
Processed 21/24 partitions; 11,326,320 keys
Processed 22/24 partitions; 11,868,043 keys
Processed 23/24 partitions; 12,410,715 keys
Processed 24/24 p

In [24]:
key_contract_features = pd.concat(key_contract_parts).sort_index()
assert key_contract_features.index.is_unique
assert key_audit["keys"] == n_source_keys
assert key_audit["represented_records"] + missing_key_records == source_shape[0]
assert key_audit["additional_entries"] == key_duplicate_extras
assert key_audit["amount_below_keys"] + key_audit["amount_equal_keys"] + key_audit["amount_above_keys"] == key_audit["amount_eligible_keys"]
assert key_audit["coverage_early_keys"] + key_audit["coverage_ontime_keys"] + key_audit["coverage_late_keys"] == key_audit["coverage_timed_keys"]
assert key_audit["coverage_timed_keys"] <= key_audit["amount_covered_keys"]
display(pd.Series(key_audit, name="All source keys").to_frame())
validation_results["Every partition: unique keys, raw-record representation, multiplicity, and eligible outcomes"] = "PASS"
del key_contract_parts, partition_codes


,All source keys
keys,12951918
represented_records,13605401
multiple_entry_keys,640905
additional_entries,653483
amount_eligible_keys,12948792
amount_below_keys,2934
amount_equal_keys,12766670
amount_above_keys,179188
amount_covered_keys,12945858
coverage_timed_keys,12945858


### Validate Real Keys Directly Against Their Source Entries

The retained examples include random contracts and missing/timing/amount boundaries.
We independently sum payment amounts, inspect schedule consistency, and calculate
coverage with small chronological arrays. Every real key in this sample is compared
with the result of the partition workflow; the invented example is not used here.


In [25]:
checked_keys = pd.concat(checked_key_parts)
assert checked_keys.index.is_unique
check_contract_ids = sorted(set(check_contract_ids))
check_records = payments.loc[payments["SK_ID_PREV"].isin(check_contract_ids)].copy()
check_records["exact_duplicate"] = exact_duplicate_members.loc[check_records.index].to_numpy()
check_records = check_records.loc[check_records[KEY].notna().all(axis=1)]


In [26]:
for key, records in check_records.groupby(KEY):
    actual = checked_keys.loc[key]
    schedule_complete = records[["DAYS_INSTALMENT", "AMT_INSTALMENT"]].notna().all().all()
    coherent = schedule_complete and records["DAYS_INSTALMENT"].nunique() == 1 and records["AMT_INSTALMENT"].nunique() == 1
    amount_complete = records["AMT_PAYMENT"].notna().all() and records["AMT_PAYMENT"].ge(0).all()
    eligible = coherent and amount_complete and records["AMT_INSTALMENT"].iloc[0] > 0 and not records["exact_duplicate"].any()
    expected_sum = records["AMT_PAYMENT"].sum(min_count=1)
    assert actual["records"] == len(records)
    assert actual["paid_count"] == records["AMT_PAYMENT"].count()
    assert actual["amount_eligible"] == eligible
    np.testing.assert_allclose(actual["recorded_payment_sum"], expected_sum, rtol=1e-12, equal_nan=True)
    expected_day = np.nan
    positive = records.loc[records["AMT_PAYMENT"].gt(0)]
    if eligible and positive["DAYS_ENTRY_PAYMENT"].notna().all():
        ordered = positive.sort_values("DAYS_ENTRY_PAYMENT")
        cumulative = np.cumsum(ordered["AMT_PAYMENT"].to_numpy(dtype="float64"))
        crossing = cumulative >= records["AMT_INSTALMENT"].iloc[0] - AMOUNT_TOLERANCE
        if crossing.any():
            expected_day = ordered["DAYS_ENTRY_PAYMENT"].to_numpy()[crossing][0]
    np.testing.assert_allclose(actual["coverage_day"], expected_day, equal_nan=True)
    if eligible:
        gap = expected_sum - records["AMT_INSTALMENT"].iloc[0]
        assert actual["amount_below"] == (gap < -AMOUNT_TOLERANCE)
        assert actual["amount_equal"] == (abs(gap) <= AMOUNT_TOLERANCE)
        assert actual["amount_above"] == (gap > AMOUNT_TOLERANCE)
print("Real installment keys checked:", len(checked_keys))
validation_results["Real source keys: cumulative amounts, missingness, and independent first coverage dates"] = "PASS"


Real installment keys checked: 564


### Combine the Contract Summaries

Both summaries are already at contract grain. A one-to-one join preserves raw
record counts and adds key-level behavior. Missing complete keys give zero key
availability counts, while amount and coverage measurements remain missing.


In [27]:
contract_features = record_features.join(key_contract_features, how="left", validate="one_to_one")
KEY_COUNT_COLUMNS = [column for column in key_contract_features if pd.api.types.is_integer_dtype(key_contract_features[column])]
contract_features[KEY_COUNT_COLUMNS] = contract_features[KEY_COUNT_COLUMNS].fillna(0).astype("int32")
assert contract_features.index.is_unique
assert len(contract_features) == n_source_contracts
assert (contract_features["key_records"] + contract_features["missing_key_records"]).eq(contract_features["records"]).all()
assert contract_features["schedule_keys"].sum() == n_source_keys
assert contract_features["additional_entries"].eq(contract_features["key_records"] - contract_features["schedule_keys"]).all()
assert contract_features["keys_with_late_records"].le(contract_features["timed_keys"]).all()
assert contract_features["severe90_records"].le(contract_features["severe60_records"]).all()
assert contract_features["severe60_records"].le(contract_features["severe30_records"]).all()
validation_results["Contract grain, raw/key distinctions, severity nesting, and joined count conservation"] = "PASS"


### Retain a Small Applicant Source Sample and Release the Full Table

We keep all source entries for a small set of mapped applicants, including
boundary-case contracts. Releasing the full raw table after its checks leaves
more memory for the applicant summaries. Unmatched contracts remain in the
contract table and will never be assigned an invented applicant mapping.


In [28]:
check_applicant_ids = previous_index.reindex(check_contract_ids)["SK_ID_CURR"].dropna().astype("int32").tolist()
no_history_applicants = previous_mapping.loc[~previous_mapping["SK_ID_CURR"].isin(previous_index.loc[matched_ids, "SK_ID_CURR"]), "SK_ID_CURR"].drop_duplicates()
check_applicant_ids += no_history_applicants.head(2).tolist()
check_applicant_ids = sorted(set(check_applicant_ids))
check_previous_mapping = previous_mapping.loc[previous_mapping["SK_ID_CURR"].isin(check_applicant_ids)].copy()
check_applicant_records = payments.loc[payments["SK_ID_PREV"].isin(check_previous_mapping["SK_ID_PREV"])].copy()
del payments, complete_key, exact_duplicate_members, checked_key_parts, checked_keys, check_records
del record_features, key_contract_features, timing_sample
gc.collect()


0

## 9. Aggregate Contract Features by Applicant

Mapping is applied to **contract summaries only**. Unmatched contracts retain
missing mapped applicant IDs in `unmatched_installment_contracts`. We verify their
record counts before selecting matched contracts for applicant aggregation.


In [29]:
contract_features["SK_ID_CURR"] = contract_features.index.map(previous_index["SK_ID_CURR"]).astype("Int32")
contract_features = contract_features.reset_index()
unmatched_installment_contracts = contract_features.loc[contract_features["SK_ID_CURR"].isna()].copy()
mapped_contracts = contract_features.loc[contract_features["SK_ID_CURR"].notna()].copy()
mapped_contracts["SK_ID_CURR"] = mapped_contracts["SK_ID_CURR"].astype("int32")
assert len(mapped_contracts) == len(matched_ids)
assert len(unmatched_installment_contracts) == len(unmatched_ids)
assert mapped_contracts["records"].sum() == n_matched_records
assert unmatched_installment_contracts["records"].sum() == n_unmatched_records
display(unmatched_installment_contracts[["SK_ID_PREV", "SK_ID_CURR", "records", "schedule_keys"]].head())
installments_grouped = mapped_contracts.groupby("SK_ID_CURR", sort=True)


,SK_ID_PREV,SK_ID_CURR,records,schedule_keys
19,1000021,<NA>,3,3
34,1000037,<NA>,2,2
38,1000041,<NA>,14,11
89,1000095,<NA>,6,6
90,1000096,<NA>,74,70


### A. Contract Counts and Coverage

Every applicant in the previous mapping gets a feature row. Counts with no
installment history are zero availability; repayment measurements remain missing.
We distinguish contracts with any known timing from history contracts with none,
and contracts with eligible key-amount comparisons from those without them.

Key mean/max describe observed version-specific history, not the contractual term.
History coverage uses all previous applications, including those that may not
have become loans. It is separate from the portfolio contract-ID match percentage.


In [30]:
for flag, source in {
    "has_timing": "timed_records", "has_amount_comparison": "amount_eligible_keys",
    "has_late_record": "late_records", "has_severe90": "severe90_records",
    "has_multiple_entries": "multiple_entry_keys", "has_version0": "version0_records",
    "has_revision_pairs": "cross_version_number_pairs"
}.items():
    mapped_contracts[flag] = mapped_contracts[source].gt(0)
mapped_contracts["has_no_timing"] = ~mapped_contracts["has_timing"]
mapped_contracts["has_no_amount_comparison"] = ~mapped_contracts["has_amount_comparison"]
mapped_contracts["has_repeated_late_records"] = mapped_contracts["late_records"].ge(2)
mapped_contracts["has_repeated_late_keys"] = mapped_contracts["keys_with_late_records"].ge(2)
mapped_contracts["has_multiple_versions"] = mapped_contracts["observed_versions"].gt(1)


In [31]:
installments_count_features = previous_mapping.groupby("SK_ID_CURR").agg(
    installments_total_previous_contracts=("SK_ID_PREV", "size")
).reset_index()
history_counts = installments_grouped.agg(
    installments_contracts_with_history=("SK_ID_PREV", "size"),
    installments_contracts_with_timing=("has_timing", "sum"),
    installments_contracts_without_timing=("has_no_timing", "sum"),
    installments_amount_comparable_contracts=("has_amount_comparison", "sum"),
    installments_contracts_without_amount_comparison=("has_no_amount_comparison", "sum"),
    installments_raw_records=("records", "sum"), installments_schedule_keys=("schedule_keys", "sum"),
    installments_keys_per_contract_mean=("schedule_keys", "mean"),
    installments_keys_per_contract_max=("schedule_keys", "max")
).reset_index()
installments_count_features = installments_count_features.merge(history_counts, on="SK_ID_CURR", how="left", validate="one_to_one")
COMPLETE_COUNT_COLUMNS = [column for column in history_counts if column not in ["SK_ID_CURR", "installments_keys_per_contract_mean", "installments_keys_per_contract_max"]]
installments_count_features[COMPLETE_COUNT_COLUMNS] = installments_count_features[COMPLETE_COUNT_COLUMNS].fillna(0).astype("int32")
installments_count_features["installments_previous_contracts_without_history"] = installments_count_features["installments_total_previous_contracts"] - installments_count_features["installments_contracts_with_history"]
installments_count_features["installments_history_coverage_ratio"] = installments_count_features["installments_contracts_with_history"] / installments_count_features["installments_total_previous_contracts"]
RATE_COLUMNS = ["installments_history_coverage_ratio"]


### B. Entry Delay Frequency and Key Recurrence

Entry rates use **known-timing payment records**. Key late rate uses keys with
at least one known entry delay. Counts of repeated late records and repeated
late keys are separate: split entries can inflate record recurrence, and versions
can still describe revisions. Neither count proves separate delinquency episodes.

We keep zero detected events when usable observations exist. If no timing exists,
event counts and rates are missing; timing availability stays explicitly zero.


In [32]:
installments_timing_features = installments_grouped.agg(
    installments_timed_records=("timed_records", "sum"),
    installments_early_records=("early_records", "sum"),
    installments_ontime_records=("ontime_records", "sum"),
    installments_late_records=("late_records", "sum"),
    installments_timed_keys=("timed_keys", "sum"),
    installments_keys_with_late_records=("keys_with_late_records", "sum"),
    installments_contracts_with_late_records=("has_late_record", "sum"),
    installments_contracts_with_repeated_late_records=("has_repeated_late_records", "sum"),
    installments_contracts_with_repeated_late_keys=("has_repeated_late_keys", "sum")
).reset_index()
known_records = installments_timing_features["installments_timed_records"].replace(0, np.nan)
for label in ["early", "ontime", "late"]:
    feature = "installments_" + label + "_records"
    installments_timing_features[feature] = installments_timing_features[feature].mask(known_records.isna())
    rate = "installments_" + label + "_record_rate"
    installments_timing_features[rate] = installments_timing_features[feature] / known_records
    RATE_COLUMNS.append(rate)
for feature in ["installments_contracts_with_late_records", "installments_contracts_with_repeated_late_records", "installments_contracts_with_repeated_late_keys"]:
    installments_timing_features[feature] = installments_timing_features[feature].mask(known_records.isna())
known_keys = installments_timing_features["installments_timed_keys"].replace(0, np.nan)
installments_timing_features["installments_keys_with_late_records"] = installments_timing_features["installments_keys_with_late_records"].mask(known_keys.isna())
installments_timing_features["installments_late_key_rate"] = installments_timing_features["installments_keys_with_late_records"] / known_keys
RATE_COLUMNS.append("installments_late_key_rate")
COMPLETE_COUNT_COLUMNS += ["installments_timed_records", "installments_timed_keys"]


### C. Delay Severity Without Cancellation by Early Entries

The signed delay mean uses total signed days / timed records. Positive-only mean
uses late days / late records. Signed maximum can be negative if all entries were
early; positive-only maximum is missing when no lateness was observed.
Severe thresholds overlap: a ≥90-day record also qualifies for ≥30 and ≥60.


In [33]:
installments_severity_features = installments_grouped.agg(
    installments_delay_sum=("delay_sum", "sum"), installments_delay_max=("delay_max", "max"),
    installments_late_delay_sum=("late_delay_sum", "sum"), installments_late_delay_max=("late_delay_max", "max"),
    installments_delay_ge30_records=("severe30_records", "sum"),
    installments_delay_ge60_records=("severe60_records", "sum"),
    installments_delay_ge90_records=("severe90_records", "sum"),
    installments_contracts_with_ge90_delay=("has_severe90", "sum")
).reset_index()
timed_denominator = installments_timing_features["installments_timed_records"].replace(0, np.nan)
late_denominator = installments_timing_features["installments_late_records"].replace(0, np.nan)
installments_severity_features["installments_delay_mean"] = installments_severity_features["installments_delay_sum"] / timed_denominator
installments_severity_features["installments_late_delay_mean"] = installments_severity_features["installments_late_delay_sum"] / late_denominator
for feature in ["installments_delay_ge30_records", "installments_delay_ge60_records", "installments_delay_ge90_records", "installments_contracts_with_ge90_delay"]:
    installments_severity_features[feature] = installments_severity_features[feature].mask(timed_denominator.isna())
installments_severity_features = installments_severity_features.drop(columns=["installments_delay_sum", "installments_late_delay_sum"])


### D. Scheduled Versus Recorded Amounts

We retain two grains: raw recorded-payment total/mean and coherent version-key
scheduled total/mean. The scheduled total can repeat revised obligations across
versions; it measures recorded schedule observations, not unique debt.

The payment-to-scheduled ratio uses paired totals from **the same eligible keys**,
so missing/zero schedules cannot create a denominator mismatch. This amount ratio
can exceed 1; it is not a bounded probability. Below/equal/above/covered key counts
use cumulative recorded payments, not single partial entries. Unknown eligibility
never becomes a confirmed shortfall or confirmed coverage.


In [34]:
installments_amount_features = installments_grouped.agg(
    installments_recorded_payment_amount_total=("recorded_payment_sum", "sum"),
    installments_version_key_scheduled_amount_total=("scheduled_key_sum", "sum"),
    installments_coherent_scheduled_keys=("coherent_amount_keys", "sum"),
    installments_amount_eligible_keys=("amount_eligible_keys", "sum"),
    installments_eligible_key_payment_amount_total=("comparison_payment_sum", "sum"),
    installments_eligible_key_scheduled_amount_total=("comparison_scheduled_sum", "sum"),
    installments_payment_below_schedule_keys=("amount_below_keys", "sum"),
    installments_payment_equal_schedule_keys=("amount_equal_keys", "sum"),
    installments_payment_above_schedule_keys=("amount_above_keys", "sum"),
    installments_payment_covering_schedule_keys=("amount_covered_keys", "sum")
).reset_index()
observed_payments = installments_grouped["payment_amount_records"].sum().to_numpy()
coherent_keys = installments_amount_features["installments_coherent_scheduled_keys"].replace(0, np.nan)
eligible_keys = installments_amount_features["installments_amount_eligible_keys"].replace(0, np.nan)
installments_amount_features["installments_recorded_payment_amount_total"] = installments_amount_features["installments_recorded_payment_amount_total"].mask(observed_payments == 0)
installments_amount_features["installments_version_key_scheduled_amount_total"] = installments_amount_features["installments_version_key_scheduled_amount_total"].mask(coherent_keys.isna())
for feature in ["installments_eligible_key_payment_amount_total", "installments_eligible_key_scheduled_amount_total", "installments_payment_below_schedule_keys", "installments_payment_equal_schedule_keys", "installments_payment_above_schedule_keys", "installments_payment_covering_schedule_keys"]:
    installments_amount_features[feature] = installments_amount_features[feature].mask(eligible_keys.isna())
installments_amount_features["installments_recorded_payment_amount_mean"] = installments_amount_features["installments_recorded_payment_amount_total"] / np.where(observed_payments > 0, observed_payments, np.nan)
installments_amount_features["installments_version_key_scheduled_amount_mean"] = installments_amount_features["installments_version_key_scheduled_amount_total"] / coherent_keys
installments_amount_features["installments_payment_to_scheduled_amount_ratio"] = installments_amount_features["installments_eligible_key_payment_amount_total"] / installments_amount_features["installments_eligible_key_scheduled_amount_total"].replace(0, np.nan)
installments_amount_features["installments_amount_coverage_rate"] = installments_amount_features["installments_payment_covering_schedule_keys"] / eligible_keys
RATE_COLUMNS.append("installments_amount_coverage_rate")
COMPLETE_COUNT_COLUMNS += ["installments_coherent_scheduled_keys", "installments_amount_eligible_keys"]


### E. Timing of Observed Cumulative Coverage

These are key-level rates among **covered keys with complete positive-payment
dates**. They are separate from entry timing rates. Keys lacking amounts, dates,
or enough recorded payments have no known coverage date. This selected population
can look different from all installment keys; we keep its denominator visible.
Positive-only coverage delay summaries avoid cancellation by early coverage.


In [35]:
installments_coverage_timing_features = installments_grouped.agg(
    installments_coverage_timed_keys=("coverage_timed_keys", "sum"),
    installments_coverage_early_keys=("coverage_early_keys", "sum"),
    installments_coverage_ontime_keys=("coverage_ontime_keys", "sum"),
    installments_coverage_late_keys=("coverage_late_keys", "sum"),
    installments_late_coverage_delay_sum=("late_coverage_delay_sum", "sum"),
    installments_late_coverage_delay_max=("late_coverage_delay_max", "max")
).reset_index()
coverage_denominator = installments_coverage_timing_features["installments_coverage_timed_keys"].replace(0, np.nan)
for label in ["early", "ontime", "late"]:
    count = "installments_coverage_" + label + "_keys"
    installments_coverage_timing_features[count] = installments_coverage_timing_features[count].mask(coverage_denominator.isna())
installments_coverage_timing_features["installments_coverage_late_rate"] = installments_coverage_timing_features["installments_coverage_late_keys"] / coverage_denominator
installments_coverage_timing_features["installments_late_coverage_delay_mean"] = installments_coverage_timing_features["installments_late_coverage_delay_sum"] / installments_coverage_timing_features["installments_coverage_late_keys"].replace(0, np.nan)
installments_coverage_timing_features = installments_coverage_timing_features.drop(columns="installments_late_coverage_delay_sum")
RATE_COLUMNS.append("installments_coverage_late_rate")
COMPLETE_COUNT_COLUMNS.append("installments_coverage_timed_keys")


### F. Multiple Entries and Schedule Versions

Multiple-entry key counts and additional entries measure recording multiplicity.
Version mean/max describe the number of distinct observed versions **per contract**;
we do not sum a version ID or treat a larger ID as a risk score. Cross-version
number pairs show potential revision overlap. Version-0 versus positive-version
late rates use their own known-entry denominators.


In [36]:
installments_version_features = installments_grouped.agg(
    installments_multiple_entry_keys=("multiple_entry_keys", "sum"),
    installments_additional_payment_entries=("additional_entries", "sum"),
    installments_contracts_with_multiple_entries=("has_multiple_entries", "sum"),
    installments_versions_per_contract_mean=("observed_versions", "mean"),
    installments_versions_per_contract_max=("observed_versions", "max"),
    installments_contracts_with_multiple_versions=("has_multiple_versions", "sum"),
    installments_version0_contracts=("has_version0", "sum"), installments_version0_keys=("version0_keys", "sum"),
    installments_version0_timed_records=("version0_timed_records", "sum"),
    installments_version0_late_records=("version0_late_records", "sum"),
    installments_positive_version_timed_records=("positive_version_timed_records", "sum"),
    installments_positive_version_late_records=("positive_version_late_records", "sum"),
    installments_cross_version_number_pairs=("cross_version_number_pairs", "sum"),
    installments_comparable_revision_pairs=("comparable_revision_pairs", "sum"),
    installments_changed_amount_revision_pairs=("changed_amount_revision_pairs", "sum"),
    installments_changed_due_revision_pairs=("changed_due_revision_pairs", "sum"),
    installments_contracts_with_revision_pairs=("has_revision_pairs", "sum")
).reset_index()
installments_version_features["installments_multiple_entry_key_rate"] = installments_version_features["installments_multiple_entry_keys"] / history_counts["installments_schedule_keys"].replace(0, np.nan)
for label in ["version0", "positive_version"]:
    denominator = installments_version_features["installments_" + label + "_timed_records"].replace(0, np.nan)
    count = "installments_" + label + "_late_records"
    installments_version_features[count] = installments_version_features[count].mask(denominator.isna())
    rate = "installments_" + label + "_late_record_rate"
    installments_version_features[rate] = installments_version_features[count] / denominator
    RATE_COLUMNS.append(rate)
RATE_COLUMNS.append("installments_multiple_entry_key_rate")
COMPLETE_COUNT_COLUMNS += [column for column in installments_version_features if column not in ["SK_ID_CURR", "installments_versions_per_contract_mean", "installments_versions_per_contract_max", "installments_version0_late_records", "installments_positive_version_late_records"] + RATE_COLUMNS]


### G. Payment Recency and Observed History

The largest signed day is the most recent observed event. Time since the latest
late entry is its negative offset; it stays missing if no late entry was recorded.
Means are **record weighted**. Due-date endpoints/span describe available history,
not a complete repayment term or calendar dates. All dates share an origin only
within each applicant, so the applicant aggregation is the appropriate grain.


In [37]:
installments_recency_features = installments_grouped.agg(
    installments_oldest_payment_day=("oldest_payment_day", "min"),
    installments_latest_payment_day=("latest_payment_day", "max"),
    installments_payment_day_sum=("payment_day_sum", "sum"),
    installments_latest_late_payment_day=("latest_late_payment_day", "max"),
    installments_oldest_scheduled_day=("oldest_due_day", "min"),
    installments_latest_scheduled_day=("latest_due_day", "max")
).reset_index()
payment_date_count = installments_grouped["payment_date_records"].sum().to_numpy()
installments_recency_features["installments_payment_day_mean"] = installments_recency_features["installments_payment_day_sum"] / np.where(payment_date_count > 0, payment_date_count, np.nan)
installments_recency_features["installments_days_since_latest_late_payment"] = -installments_recency_features["installments_latest_late_payment_day"]
installments_recency_features["installments_scheduled_history_span_days"] = installments_recency_features["installments_latest_scheduled_day"] - installments_recency_features["installments_oldest_scheduled_day"]
installments_recency_features = installments_recency_features.drop(columns="installments_payment_day_sum")


### H. Missingness and Availability

Missing dates/amounts are counted separately from genuine zero values. No usable
raw payment amount and no eligible cumulative amount comparison are different:
the second also requires schedule consistency and complete payments at the key.
Incomplete/conflicting schedule keys and zero schedules stay visible here.

القيمة الناقصة مش صفر. القسط اللي ناقصه بيانات ما نقدرش نحكم إنه اتغطى أو إنه ناقص دفع.


In [38]:
mapped_contracts["has_missing_payment_date"] = mapped_contracts["missing_payment_dates"].gt(0)
mapped_contracts["has_missing_payment_amount"] = mapped_contracts["missing_payment_amounts"].gt(0)
mapped_contracts["has_no_payment_amount"] = mapped_contracts["payment_amount_records"].eq(0)
installments_availability_features = installments_grouped.agg(
    installments_missing_payment_date_records=("missing_payment_dates", "sum"),
    installments_missing_scheduled_date_records=("missing_due_dates", "sum"),
    installments_missing_timing_records=("missing_timing_records", "sum"),
    installments_missing_payment_amount_records=("missing_payment_amounts", "sum"),
    installments_missing_scheduled_amount_records=("missing_scheduled_amounts", "sum"),
    installments_observed_payment_amount_records=("payment_amount_records", "sum"),
    installments_contracts_with_missing_payment_dates=("has_missing_payment_date", "sum"),
    installments_contracts_with_missing_payment_amounts=("has_missing_payment_amount", "sum"),
    installments_contracts_without_payment_amounts=("has_no_payment_amount", "sum"),
    installments_incomplete_payment_keys=("incomplete_payment_keys", "sum"),
    installments_incomplete_schedule_keys=("incomplete_schedule_keys", "sum"),
    installments_inconsistent_schedule_keys=("inconsistent_schedule_keys", "sum"),
    installments_zero_scheduled_amount_keys=("zero_schedule_keys", "sum"),
    installments_missing_key_records=("missing_key_records", "sum")
).reset_index()
COMPLETE_COUNT_COLUMNS += [column for column in installments_availability_features if column != "SK_ID_CURR"]


## 10. Combine Applicant-Level Feature Groups

Each group is already one row per applicant. We use one-to-one merges, as in the
Bureau notebook. Only explicit availability/history/multiplicity counts are zero
for applicants without installment history. Measurements, dates, eligible outcome
counts, and rates retain missingness when their required observations do not exist.


In [39]:
feature_groups = {
    "Contract counts and history coverage": installments_count_features,
    "Entry delay frequency and key recurrence": installments_timing_features,
    "Entry delay severity": installments_severity_features,
    "Version-key scheduled and recorded amounts": installments_amount_features,
    "Observed cumulative coverage timing": installments_coverage_timing_features,
    "Multiple entries and calendar versions": installments_version_features,
    "Relative-day recency and history": installments_recency_features,
    "Missingness and data availability": installments_availability_features
}
for name, group in feature_groups.items():
    assert group["SK_ID_CURR"].is_unique, name
    assert len(group) == (n_applicants if group is installments_count_features else n_mapped_history_applicants), name
installments_features = installments_count_features.copy()
for group in list(feature_groups.values())[1:]:
    installments_features = installments_features.merge(group, on="SK_ID_CURR", how="left", validate="one_to_one")
installments_features = installments_features.sort_values("SK_ID_CURR").reset_index(drop=True)
no_history = installments_features["installments_contracts_with_history"].eq(0)
installments_features.loc[no_history, COMPLETE_COUNT_COLUMNS] = 0
for column in COMPLETE_COUNT_COLUMNS + ["installments_total_previous_contracts", "installments_previous_contracts_without_history"]:
    installments_features[column] = installments_features[column].astype("int32")
print("Shape:", installments_features.shape)
print("Unique applicants:", installments_features["SK_ID_CURR"].nunique())
print("Applicants without mapped installment history:", int(no_history.sum()))
installments_features.head()


Shape: (338857, 97)
Unique applicants: 338857
Applicants without mapped installment history: 1922


,SK_ID_CURR,installments_total_previous_contracts,installments_contracts_with_history,installments_contracts_with_timing,installments_contracts_without_timing,installments_amount_comparable_contracts,installments_contracts_without_amount_comparison,installments_raw_records,installments_schedule_keys,installments_keys_per_contract_mean,...,installments_missing_scheduled_amount_records,installments_observed_payment_amount_records,installments_contracts_with_missing_payment_dates,installments_contracts_with_missing_payment_amounts,installments_contracts_without_payment_amounts,installments_incomplete_payment_keys,installments_incomplete_schedule_keys,installments_inconsistent_schedule_keys,installments_zero_scheduled_amount_keys,installments_missing_key_records
0,100001,1,1,1,0,1,0,4,4,4.000000,...,0,4,0,0,0,0,0,0,0,0
1,100002,1,1,1,0,1,0,19,19,19.000000,...,0,19,0,0,0,0,0,0,0,0
2,100003,3,3,3,0,3,0,25,25,8.333333,...,0,25,0,0,0,0,0,0,0,0
3,100004,1,1,1,0,1,0,3,3,3.000000,...,0,3,0,0,0,0,0,0,0,0
4,100005,2,1,1,0,1,0,9,9,9.000000,...,0,9,0,0,0,0,0,0,0,0


## 11. Validation

We check exact applicant population, conserved contracts/records/keys, timing
classes, severe thresholds, amount comparisons, versions, and rate formulas.
Missing measurements are validated against their availability denominators,
rather than replaced with zeros to make the table look complete.


In [40]:
assert installments_features["SK_ID_CURR"].notna().all()
assert installments_features["SK_ID_CURR"].is_unique
assert len(installments_features) == n_applicants
np.testing.assert_array_equal(installments_features["SK_ID_CURR"], np.sort(previous_mapping["SK_ID_CURR"].unique()))
assert installments_features.columns.is_unique
assert all(column.startswith("installments_") for column in installments_features if column != "SK_ID_CURR")
assert "TARGET" not in installments_features.columns
assert installments_features["installments_contracts_with_history"].sum() == len(matched_ids)
assert installments_features["installments_total_previous_contracts"].sum() == len(previous_mapping)
assert installments_features["installments_previous_contracts_without_history"].sum() == len(previous_without_history)
assert installments_features["installments_raw_records"].sum() == n_matched_records
assert installments_features["installments_schedule_keys"].sum() == mapped_contracts["schedule_keys"].sum()
assert (installments_features["installments_contracts_with_timing"] + installments_features["installments_contracts_without_timing"]).eq(installments_features["installments_contracts_with_history"]).all()
assert (installments_features["installments_amount_comparable_contracts"] + installments_features["installments_contracts_without_amount_comparison"]).eq(installments_features["installments_contracts_with_history"]).all()
assert (installments_features["installments_additional_payment_entries"] + installments_features["installments_schedule_keys"] + installments_features["installments_missing_key_records"]).eq(installments_features["installments_raw_records"]).all()
validation_results["Unique final applicant grain, mapping population, and record/key/contract conservation"] = "PASS"


In [41]:
for feature, source in {
    "installments_early_records": "early_records", "installments_ontime_records": "ontime_records",
    "installments_late_records": "late_records", "installments_keys_with_late_records": "keys_with_late_records",
    "installments_delay_ge30_records": "severe30_records", "installments_delay_ge60_records": "severe60_records",
    "installments_delay_ge90_records": "severe90_records", "installments_multiple_entry_keys": "multiple_entry_keys",
    "installments_payment_below_schedule_keys": "amount_below_keys", "installments_payment_equal_schedule_keys": "amount_equal_keys",
    "installments_payment_above_schedule_keys": "amount_above_keys", "installments_payment_covering_schedule_keys": "amount_covered_keys",
    "installments_coverage_late_keys": "coverage_late_keys", "installments_cross_version_number_pairs": "cross_version_number_pairs"
}.items():
    assert installments_features[feature].sum(min_count=1) == mapped_contracts[source].sum(min_count=1), feature
assert installments_features[["installments_early_records", "installments_ontime_records", "installments_late_records"]].sum(axis=1).eq(installments_features["installments_timed_records"]).all()
eligible = installments_features["installments_amount_eligible_keys"]
assert installments_features[["installments_payment_below_schedule_keys", "installments_payment_equal_schedule_keys", "installments_payment_above_schedule_keys"]].sum(axis=1).eq(eligible).all()
assert installments_features[["installments_payment_equal_schedule_keys", "installments_payment_above_schedule_keys"]].sum(axis=1).eq(installments_features["installments_payment_covering_schedule_keys"].fillna(0)).all()
assert installments_features[["installments_coverage_early_keys", "installments_coverage_ontime_keys", "installments_coverage_late_keys"]].sum(axis=1).eq(installments_features["installments_coverage_timed_keys"]).all()
assert installments_features["installments_coverage_timed_keys"].le(installments_features["installments_payment_covering_schedule_keys"].fillna(0)).all()
validation_results["Applicant timing, thresholds, multiplicity, amount outcomes, and versions conserved"] = "PASS"


### Independent Applicant Checks Against Raw Entries

For retained applicants we select previous IDs from the mapping, then inspect
all of their source records. We check record-weighted timing/amount statistics,
recency, key counts, and scheduled/comparison sums. Applicants without history
are included explicitly, so their zero availability and missing measurements are tested.


In [42]:
check_features = installments_features.set_index("SK_ID_CURR").loc[check_applicant_ids].astype("float64")
for applicant_id in check_applicant_ids:
    previous_ids = check_previous_mapping.loc[check_previous_mapping["SK_ID_CURR"].eq(applicant_id), "SK_ID_PREV"]
    records = check_applicant_records.loc[check_applicant_records["SK_ID_PREV"].isin(previous_ids)]
    actual = check_features.loc[applicant_id]
    delays = records["days_late"].dropna().to_numpy(dtype="float64")
    positive_delays = delays[delays > 0]
    dates = records["DAYS_ENTRY_PAYMENT"].dropna().to_numpy(dtype="float64")
    paid = records["AMT_PAYMENT"].dropna().to_numpy(dtype="float64")
    assert actual["installments_raw_records"] == len(records)
    assert actual["installments_contracts_with_history"] == records["SK_ID_PREV"].nunique()
    assert actual["installments_timed_records"] == len(delays)
    expected = {
        "installments_delay_mean": delays.mean() if len(delays) else np.nan,
        "installments_delay_max": delays.max() if len(delays) else np.nan,
        "installments_late_delay_mean": positive_delays.mean() if len(positive_delays) else np.nan,
        "installments_late_delay_max": positive_delays.max() if len(positive_delays) else np.nan,
        "installments_late_record_rate": (delays > 0).mean() if len(delays) else np.nan,
        "installments_recorded_payment_amount_total": paid.sum() if len(paid) else np.nan,
        "installments_recorded_payment_amount_mean": paid.mean() if len(paid) else np.nan,
        "installments_payment_day_mean": dates.mean() if len(dates) else np.nan,
        "installments_latest_payment_day": dates.max() if len(dates) else np.nan,
        "installments_oldest_payment_day": dates.min() if len(dates) else np.nan
    }
    for feature, value in expected.items():
        np.testing.assert_allclose(actual[feature], value, rtol=1e-12, equal_nan=True)
    for threshold in [30, 60, 90]:
        np.testing.assert_allclose(actual[f"installments_delay_ge{threshold}_records"], (delays >= threshold).sum() if len(delays) else np.nan, equal_nan=True)
    latest_late_day = records.loc[records["days_late"].gt(0), "DAYS_ENTRY_PAYMENT"].max()
    np.testing.assert_allclose(actual["installments_latest_late_payment_day"], latest_late_day, equal_nan=True)
    np.testing.assert_allclose(actual["installments_days_since_latest_late_payment"], -latest_late_day, equal_nan=True)


In [43]:
for applicant_id in check_applicant_ids:
    previous_ids = check_previous_mapping.loc[check_previous_mapping["SK_ID_CURR"].eq(applicant_id), "SK_ID_PREV"]
    records = check_applicant_records.loc[check_applicant_records["SK_ID_PREV"].isin(previous_ids)]
    key_records = records.loc[records[KEY].notna().all(axis=1)]
    actual = check_features.loc[applicant_id]
    expected_keys = key_records.groupby(KEY).ngroups
    assert actual["installments_schedule_keys"] == expected_keys
    schedules, comparable_schedules, comparable_payments = [], [], []
    for key, entries in key_records.groupby(KEY):
        coherent = entries[["DAYS_INSTALMENT", "AMT_INSTALMENT"]].notna().all().all() and entries["DAYS_INSTALMENT"].nunique() == 1 and entries["AMT_INSTALMENT"].nunique() == 1
        if coherent and entries["AMT_INSTALMENT"].iloc[0] >= 0:
            schedules.append(entries["AMT_INSTALMENT"].iloc[0])
        eligible = coherent and entries["AMT_INSTALMENT"].iloc[0] > 0 and entries["AMT_PAYMENT"].notna().all() and entries["AMT_PAYMENT"].ge(0).all() and not entries.duplicated(required_columns, keep=False).any()
        if eligible:
            comparable_schedules.append(entries["AMT_INSTALMENT"].iloc[0])
            comparable_payments.append(entries["AMT_PAYMENT"].sum())
    assert actual["installments_amount_eligible_keys"] == len(comparable_schedules)
    np.testing.assert_allclose(actual["installments_version_key_scheduled_amount_total"], sum(schedules) if schedules else np.nan, rtol=1e-12, equal_nan=True)
    np.testing.assert_allclose(actual["installments_eligible_key_scheduled_amount_total"], sum(comparable_schedules) if comparable_schedules else np.nan, rtol=1e-12, equal_nan=True)
    np.testing.assert_allclose(actual["installments_eligible_key_payment_amount_total"], sum(comparable_payments) if comparable_payments else np.nan, rtol=1e-12, equal_nan=True)
validation_results["Independent raw-source applicant timing, recency, key, and cumulative amount checks"] = "PASS"
print("Applicants checked directly:", len(check_applicant_ids))
del check_features, check_applicant_records, check_previous_mapping


Applicants checked directly: 26


### Rate Formulas and Numeric Limits

All frequency/coverage rates use a documented eligible denominator and lie in
0–1. The payment amount ratio can exceed 1 and is checked separately. Signed
delays may be negative; positive-only delays and elapsed days cannot be negative.
Zero denominators produce missing values, never infinity.


In [44]:
rate_definitions = {
    "installments_history_coverage_ratio": ("installments_contracts_with_history", "installments_total_previous_contracts"),
    "installments_early_record_rate": ("installments_early_records", "installments_timed_records"),
    "installments_ontime_record_rate": ("installments_ontime_records", "installments_timed_records"),
    "installments_late_record_rate": ("installments_late_records", "installments_timed_records"),
    "installments_late_key_rate": ("installments_keys_with_late_records", "installments_timed_keys"),
    "installments_amount_coverage_rate": ("installments_payment_covering_schedule_keys", "installments_amount_eligible_keys"),
    "installments_coverage_late_rate": ("installments_coverage_late_keys", "installments_coverage_timed_keys"),
    "installments_multiple_entry_key_rate": ("installments_multiple_entry_keys", "installments_schedule_keys"),
    "installments_version0_late_record_rate": ("installments_version0_late_records", "installments_version0_timed_records"),
    "installments_positive_version_late_record_rate": ("installments_positive_version_late_records", "installments_positive_version_timed_records")
}
assert set(rate_definitions) == set(RATE_COLUMNS)
for feature, (numerator, denominator) in rate_definitions.items():
    expected = installments_features[numerator] / installments_features[denominator].replace(0, np.nan)
    np.testing.assert_allclose(installments_features[feature], expected, equal_nan=True)
    assert installments_features[feature].dropna().between(0, 1).all(), feature
    assert installments_features[feature].isna().eq(installments_features[denominator].eq(0)).all(), feature
ratio = installments_features["installments_payment_to_scheduled_amount_ratio"]
np.testing.assert_allclose(ratio, installments_features["installments_eligible_key_payment_amount_total"] / installments_features["installments_eligible_key_scheduled_amount_total"].replace(0, np.nan), equal_nan=True)
assert ratio.dropna().ge(0).all()
for feature in ["installments_late_delay_mean", "installments_late_delay_max", "installments_late_coverage_delay_mean", "installments_late_coverage_delay_max", "installments_days_since_latest_late_payment"]:
    assert installments_features[feature].dropna().ge(0).all(), feature
assert not np.isinf(installments_features.to_numpy(dtype="float64", na_value=np.nan)).any()
validation_results["Eligible rate formulas, bounded rates, amount ratio, positive-only delays, and no infinities"] = "PASS"


### Missingness Must Match Availability

- No history: zero observation counts; missing history-size means and repayment measurements.
- No timing: missing entry outcomes, delay statistics, and timing rates.
- No late entries: missing positive-only delay mean/max and last-late recency.
- No eligible amount keys: missing cumulative comparisons and their amount ratio.
- No observed payment amounts: missing raw payment total/mean, not a zero payment total.
- No covered/date-complete keys: missing coverage timing outcomes and rate.
- Incomplete/conflicting schedules remain explicit availability counts.

These are observed-data limits, not proof of good or bad lifetime repayment.


In [45]:
missing_summary = pd.DataFrame({
    "missing_applicants": installments_features.isna().sum(),
    "missing_percent": (installments_features.isna().mean() * 100).round(2)
}).sort_values("missing_applicants", ascending=False)
display(missing_summary.loc[missing_summary["missing_applicants"].gt(0)])
assert installments_features[COMPLETE_COUNT_COLUMNS].notna().all().all()
assert installments_features["installments_missing_timing_records"].eq(installments_features["installments_raw_records"] - installments_features["installments_timed_records"]).all()
assert installments_features["installments_missing_payment_amount_records"].eq(installments_features["installments_raw_records"] - installments_features["installments_observed_payment_amount_records"]).all()
for feature in ["installments_keys_per_contract_mean", "installments_keys_per_contract_max", "installments_versions_per_contract_mean", "installments_versions_per_contract_max"]:
    assert installments_features[feature].isna().eq(no_history).all(), feature
no_timing = installments_features["installments_timed_records"].eq(0)
for feature in ["installments_delay_mean", "installments_delay_max", "installments_early_records", "installments_ontime_records", "installments_late_records", "installments_contracts_with_late_records", "installments_contracts_with_repeated_late_records", "installments_delay_ge90_records"]:
    assert installments_features[feature].isna().eq(no_timing).all(), feature
no_late = installments_features["installments_late_records"].fillna(0).eq(0)
for feature in ["installments_late_delay_mean", "installments_late_delay_max", "installments_latest_late_payment_day", "installments_days_since_latest_late_payment"]:
    assert installments_features[feature].isna().eq(no_late).all(), feature


,missing_applicants,missing_percent
installments_version0_late_records,276584,81.62
installments_version0_late_record_rate,276584,81.62
installments_late_coverage_delay_mean,164761,48.62
installments_late_coverage_delay_max,164761,48.62
installments_latest_late_payment_day,164708,48.61
installments_days_since_latest_late_payment,164708,48.61
installments_late_delay_mean,164708,48.61
installments_late_delay_max,164708,48.61
installments_positive_version_late_records,4606,1.36
installments_positive_version_late_record_rate,4606,1.36


In [46]:
null_rules = {
    "installments_recorded_payment_amount_total": "installments_observed_payment_amount_records",
    "installments_recorded_payment_amount_mean": "installments_observed_payment_amount_records",
    "installments_version_key_scheduled_amount_total": "installments_coherent_scheduled_keys",
    "installments_version_key_scheduled_amount_mean": "installments_coherent_scheduled_keys",
    "installments_eligible_key_payment_amount_total": "installments_amount_eligible_keys",
    "installments_eligible_key_scheduled_amount_total": "installments_amount_eligible_keys",
    "installments_payment_to_scheduled_amount_ratio": "installments_amount_eligible_keys",
    "installments_payment_below_schedule_keys": "installments_amount_eligible_keys",
    "installments_payment_equal_schedule_keys": "installments_amount_eligible_keys",
    "installments_payment_above_schedule_keys": "installments_amount_eligible_keys",
    "installments_payment_covering_schedule_keys": "installments_amount_eligible_keys",
    "installments_coverage_early_keys": "installments_coverage_timed_keys",
    "installments_coverage_ontime_keys": "installments_coverage_timed_keys",
    "installments_coverage_late_keys": "installments_coverage_timed_keys"
}
for feature, count in null_rules.items():
    assert installments_features[feature].isna().eq(installments_features[count].eq(0)).all(), feature
for label in ["version0", "positive_version"]:
    assert installments_features["installments_" + label + "_late_records"].isna().eq(installments_features["installments_" + label + "_timed_records"].eq(0)).all()
validation_results["Feature nulls and missing counts agree with timing, amount, version, and coverage availability"] = "PASS"


## 12. Save the Applicant Features to Parquet

We use `data/interim/installments_features.parquet`, following the other feature
outputs. No index is saved. A new file is created exclusively; on reruns an existing
identical output is verified and preserved. A different existing output stops
for review instead of being overwritten.


In [47]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
if OUTPUT_PATH.exists():
    existing_installments = pd.read_parquet(OUTPUT_PATH)
    pd.testing.assert_frame_equal(existing_installments, installments_features, check_exact=True)
    print("Existing identical output verified and preserved:", OUTPUT_PATH)
    del existing_installments
else:
    with OUTPUT_PATH.open("xb") as output_file:
        installments_features.to_parquet(output_file, index=False)
    print("Saved:", OUTPUT_PATH)


Saved: e:\Depi Notes Data Engineering\DEPI project R5\ProJect\data\interim\installments_features.parquet


## 13. Reload and Verify the Output

Successful reading alone is insufficient. We compare shape, IDs, column order,
dtypes, exact values, and nulls, then verify both raw CSV contents with SHA-256.


In [48]:
saved_installments = pd.read_parquet(OUTPUT_PATH)
print("Reloaded shape:", saved_installments.shape)
print("Unique IDs:", saved_installments["SK_ID_CURR"].is_unique)
print("Duplicate IDs:", saved_installments["SK_ID_CURR"].duplicated().sum())
assert saved_installments.shape == installments_features.shape
assert saved_installments["SK_ID_CURR"].is_unique
assert saved_installments["SK_ID_CURR"].notna().all()
pd.testing.assert_frame_equal(saved_installments, installments_features, check_exact=True)
pd.testing.assert_series_equal(saved_installments.isna().sum(), installments_features.isna().sum())
validation_results["Parquet reload exactly preserves shape, IDs, dtypes, values, and nulls"] = "PASS"
saved_installments.head()


Reloaded shape: (338857, 97)
Unique IDs: True
Duplicate IDs: 0


,SK_ID_CURR,installments_total_previous_contracts,installments_contracts_with_history,installments_contracts_with_timing,installments_contracts_without_timing,installments_amount_comparable_contracts,installments_contracts_without_amount_comparison,installments_raw_records,installments_schedule_keys,installments_keys_per_contract_mean,...,installments_missing_scheduled_amount_records,installments_observed_payment_amount_records,installments_contracts_with_missing_payment_dates,installments_contracts_with_missing_payment_amounts,installments_contracts_without_payment_amounts,installments_incomplete_payment_keys,installments_incomplete_schedule_keys,installments_inconsistent_schedule_keys,installments_zero_scheduled_amount_keys,installments_missing_key_records
0,100001,1,1,1,0,1,0,4,4,4.000000,...,0,4,0,0,0,0,0,0,0,0
1,100002,1,1,1,0,1,0,19,19,19.000000,...,0,19,0,0,0,0,0,0,0,0
2,100003,3,3,3,0,3,0,25,25,8.333333,...,0,25,0,0,0,0,0,0,0,0
3,100004,1,1,1,0,1,0,3,3,3.000000,...,0,3,0,0,0,0,0,0,0,0
4,100005,2,1,1,0,1,0,9,9,9.000000,...,0,9,0,0,0,0,0,0,0,0


In [49]:
for path, original_sha256 in raw_sha256_before.items():
    with path.open("rb") as raw_file:
        current_sha256 = hashlib.file_digest(raw_file, "sha256").hexdigest()
    assert current_sha256 == original_sha256, f"Raw CSV changed: {path.name}"
validation_results["Both raw input CSV SHA-256 fingerprints unchanged"] = "PASS"
display(saved_installments.dtypes.value_counts().rename("columns").to_frame())
display(pd.Series(validation_results, name="Result").to_frame())


,columns
float64,46
int32,42
float32,9


,Result
"Source schema, IDs, numeric domains, and application-time offsets",PASS
"Mapping conflicts, source-ID agreement, and matched/unmatched conservation",PASS
Source entry delay formula and eligible timing classification,PASS
"Illustrative split-payment, late-extra-entry, missingness, and version boundaries",PASS
Unique record-summary contract grain and source timing/amount conservation,PASS
"Every partition: unique keys, raw-record representation, multiplicity, and eligible outcomes",PASS
"Real source keys: cumulative amounts, missingness, and independent first coverage dates",PASS
"Contract grain, raw/key distinctions, severity nesting, and joined count conservation",PASS
"Unique final applicant grain, mapping population, and record/key/contract conservation",PASS
"Applicant timing, thresholds, multiplicity, amount outcomes, and versions conserved",PASS


## 14. Final Summary

This report is generated from the executed source and feature tables. Source
applicant coverage, mapped-history applicant coverage, and the final mapping
population are reported separately.


In [50]:
n_engineered_features = saved_installments.shape[1] - 1
category_summary = "\n".join(f"| {name} | {group.shape[1] - 1} |" for name, group in feature_groups.items())
validation_summary = "\n".join(f"| {check} | {result} |" for check, result in validation_results.items())
display(Markdown(f"""
| Summary | Result |
|---|---|
| Source dimensions / raw payment records | {source_shape[0]:,} records × {source_shape[1]} columns |
| Unique previous contracts | {n_source_contracts:,} |
| Mapped / unmatched contracts | {len(matched_ids):,} / {len(unmatched_ids):,} |
| Contract ID match coverage | {len(matched_ids) / n_source_contracts:.2%} |
| Matched / unmatched raw payment records | {n_matched_records:,} / {n_unmatched_records:,} |
| Source observed applicants (audit only) | {f'{n_source_applicants:,}' if n_source_applicants is not None else 'Column absent'} |
| Applicants with mapped installment history | {n_mapped_history_applicants:,} |
| Applicants in final mapping population | {n_applicants:,} |
| Applicants without mapped installment history | {int(no_history.sum()):,} |
| Complete version-specific installment keys | {n_source_keys:,} |
| Mapped installment keys in output | {int(saved_installments['installments_schedule_keys'].sum()):,} |
| Multiple-entry keys / share of complete keys | {key_audit['multiple_entry_keys']:,} / {key_audit['multiple_entry_keys'] / n_source_keys:.2%} |
| Additional entries beyond the first | {key_duplicate_extras:,} |
| Exact duplicate extras retained / missing-key records | {exact_duplicate_extras:,} / {missing_key_records:,} |
| Eligible cumulative amount comparison keys | {key_audit['amount_eligible_keys']:,} |
| Eligible cumulative below / equal / above keys | {key_audit['amount_below_keys']:,} / {key_audit['amount_equal_keys']:,} / {key_audit['amount_above_keys']:,} |
| Covered keys with observed coverage timing | {key_audit['coverage_timed_keys']:,} |
| Late entries despite non-late coverage keys | {key_audit['late_entries_despite_nonlate_coverage']:,} |
| Contract/number pairs across versions | {key_audit['cross_version_number_pairs']:,} |
| Contracts with multiple observed versions | {int(contract_features['observed_versions'].gt(1).sum()):,} |
| Source early / on-time / late entries | {source_timing_audit['early_records']:,} / {source_timing_audit['ontime_records']:,} / {source_timing_audit['late_records']:,} |
| Engineered features, excluding SK_ID_CURR | {n_engineered_features} |
| Final and reloaded shape | {saved_installments.shape} |
| Output | `{OUTPUT_PATH.relative_to(REPO_ROOT)}` |
| Raw inputs unchanged | Both SHA-256 fingerprints matched |

| Feature category | Features |
|---|---|
{category_summary}

| Validation | Result |
|---|---|
{validation_summary}
"""))
assert sum(group.shape[1] - 1 for group in feature_groups.values()) == n_engineered_features



| Summary | Result |
|---|---|
| Source dimensions / raw payment records | 13,605,401 records × 8 columns |
| Unique previous contracts | 997,752 |
| Mapped / unmatched contracts | 958,905 / 38,847 |
| Contract ID match coverage | 96.11% |
| Matched / unmatched raw payment records | 12,354,575 / 1,250,826 |
| Source observed applicants (audit only) | 339,587 |
| Applicants with mapped installment history | 336,935 |
| Applicants in final mapping population | 338,857 |
| Applicants without mapped installment history | 1,922 |
| Complete version-specific installment keys | 12,951,918 |
| Mapped installment keys in output | 11,730,066 |
| Multiple-entry keys / share of complete keys | 640,905 / 4.95% |
| Additional entries beyond the first | 653,483 |
| Exact duplicate extras retained / missing-key records | 0 / 0 |
| Eligible cumulative amount comparison keys | 12,948,792 |
| Eligible cumulative below / equal / above keys | 2,934 / 12,766,670 / 179,188 |
| Covered keys with observed coverage timing | 12,945,858 |
| Late entries despite non-late coverage keys | 66 |
| Contract/number pairs across versions | 89,897 |
| Contracts with multiple observed versions | 399,399 |
| Source early / on-time / late entries | 9,309,477 / 3,146,350 / 1,146,669 |
| Engineered features, excluding SK_ID_CURR | 96 |
| Final and reloaded shape | (338857, 97) |
| Output | `data\interim\installments_features.parquet` |
| Raw inputs unchanged | Both SHA-256 fingerprints matched |

| Feature category | Features |
|---|---|
| Contract counts and history coverage | 12 |
| Entry delay frequency and key recurrence | 13 |
| Entry delay severity | 8 |
| Version-key scheduled and recorded amounts | 14 |
| Observed cumulative coverage timing | 7 |
| Multiple entries and calendar versions | 20 |
| Relative-day recency and history | 8 |
| Missingness and data availability | 14 |

| Validation | Result |
|---|---|
| Source schema, IDs, numeric domains, and application-time offsets | PASS |
| Mapping conflicts, source-ID agreement, and matched/unmatched conservation | PASS |
| Source entry delay formula and eligible timing classification | PASS |
| Illustrative split-payment, late-extra-entry, missingness, and version boundaries | PASS |
| Unique record-summary contract grain and source timing/amount conservation | PASS |
| Every partition: unique keys, raw-record representation, multiplicity, and eligible outcomes | PASS |
| Real source keys: cumulative amounts, missingness, and independent first coverage dates | PASS |
| Contract grain, raw/key distinctions, severity nesting, and joined count conservation | PASS |
| Unique final applicant grain, mapping population, and record/key/contract conservation | PASS |
| Applicant timing, thresholds, multiplicity, amount outcomes, and versions conserved | PASS |
| Independent raw-source applicant timing, recency, key, and cumulative amount checks | PASS |
| Eligible rate formulas, bounded rates, amount ratio, positive-only delays, and no infinities | PASS |
| Feature nulls and missing counts agree with timing, amount, version, and coverage availability | PASS |
| Parquet reload exactly preserves shape, IDs, dtypes, values, and nulls | PASS |
| Both raw input CSV SHA-256 fingerprints unchanged | PASS |


### Decisions and Limitations

- **Grain:** Raw entries, version-specific keys, previous contracts, and applicants
  are distinct. Multiple entries are retained and their payments can collectively
  cover a schedule. A repeated late entry does not necessarily represent another
  late installment. Distinct late version keys can still refer to revised obligations.
- **Versions:** Version 0 is a credit-card calendar and remains included with its
  own timing exposure. Dates, not installment-number order, determine delay and
  coverage timing. Versions stay separate; neither the maximum version nor number
  continuity establishes chronology, distress, or a unique contractual obligation.
- **Amounts:** Schedule amounts are counted once per coherent version key. Totals
  across these keys are recorded schedule observations, not independent debt.
  Only coherent positive schedules with complete nonnegative payments and no exact
  duplicates receive cumulative comparisons. `1e-6` is a numerical tolerance in
  dataset units, not a forgiveness amount or assumed monetary rounding rule.
- **Coverage timing:** First observed cumulative coverage uses dated positive
  payments. It does not prove final settlement, correct payment allocation, or
  complete observation. Missing/insufficient keys have no known coverage date;
  excluding them creates a selected timing population that must be interpreted
  alongside eligible-key and missingness counts.
- **Missingness and exposure:** Missing data are not zero payments or on-time
  entries. Counts reflect detected behavior within available evidence; longer
  histories and split entries increase detection opportunity. Means/rates are
  explicitly record weighted or key weighted, while history/version means give
  each observed contract equal weight. Thresholds are analytical, not default labels.
- **Mapping:** Applicant features use only previous-application IDs. Unmatched
  contract summaries remain in `unmatched_installment_contracts`, even when the
  raw CSV supplies an applicant ID. Final rows cover the full previous-mapping
  applicant population. Applicants absent from that mapping require a later
  applicant-table join. History coverage includes all previous applications,
  some of which may not have created repayment obligations.
- **Reproducibility and leakage:** All used dates are nonpositive application-
  relative offsets; no calendar dates or target information are derived. Source
  availability at scoring time and allocation/version semantics still need
  confirmation for production use. All raw rows remain represented; anomalous
  keys are excluded only from comparisons requiring their missing context.
  Raw CSVs and existing notebooks/feature outputs remain unchanged.
